# Fixed Price Runs: `fixed_path` and `fixed_rate` pricing modes

By default, NZUpy **optimises** the price change rate to find the carbon price path that balances supply and demand. However, we can also use two alternative pricing modes that skip the optimiser entirely:

| Mode | What it does |
|---|---|
| `optimised` | Default — finds the price growth rate that clears the market with minimal difference between supply and demand from start to end year |
| `fixed_path` | User supplies an explicit year-by-year price series |
| `fixed_rate` | User supplies a scalar annual growth rate; prices grow from the 2024 starting price at that rate |

These modes are useful for:
- **Sensitivity analysis**: hold the price fixed and study how supply/demand components respond
- **Policy scenarios**: model the effect of an explicit price scenario (e.g., Climate Change Commission price path)
- **Comparison runs**: contrast emissions impacts and stockpile dynamics under different price path assumptions

Pricing modes are set via `set_mode()` and price data is supplied via `fill()`.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from pathlib import Path
import sys
import os

# Add project root to path
project_root = Path().absolute().parent
sys.path.insert(0, str(project_root))

from model.core.base_model import NZUpy

data_dir = project_root / "data"
output_dir = project_root / "examples" / "outputs" / "05_fixed_price_runs"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Initialise model with three scenarios
NZU = NZUpy(data_dir=data_dir)
NZU.define_time(2024, 2050)
NZU.define_scenarios(['Optimised', 'Fixed Path', 'Fixed Rate'])
NZU.allocate()
NZU.fill_defaults()

## 2. Fixed price path (`fixed_path`)

Supply a year-by-year price series that the model will use directly.

Here we define a simple linear path rising from \$50 to \$90 over the modelling period.

In [ ]:
years = NZU.years  # 2024–2050

# Linear price path: $50 in 2024, +$1.54/yr to reach ~$90 in 2050
price_path = pd.Series(
    {y: 50.0 + (y - 2024) * (90.0 - 50.0) / (2050 - 2024) for y in range(2024, 2051)}
)

print("Price path (first 5 years):")
print(price_path.head().round(2))

In [ ]:
# Set fixed_path mode and supply the price series
NZU.set_mode('pricing_mode', 'fixed_path', scenario='Fixed Path')
NZU.fill('price_path', price_path, scenario='Fixed Path')

## 3. Fixed price change rate (`fixed_rate`)

Supply a single annual growth rate. Prices grow from the model's 2024 starting price at this rate each year.

Here we use 3% per year.

In [ ]:
# Set fixed_rate mode and supply the growth rate
NZU.set_mode('pricing_mode', 'fixed_rate', scenario='Fixed Rate')
NZU.fill('price_change_rate', 0.03, scenario='Fixed Rate')

## 4. Run all three scenarios

In [ ]:
results = NZU.run()

## 5. Results: price trajectories

Let's compare the three price paths.

In [ ]:
# Extract prices for model years from each scenario
prices_opt    = NZU.results['Optimised']['prices'].loc[years]
prices_fp     = NZU.results['Fixed Path']['prices'].loc[years]
prices_fr     = NZU.results['Fixed Rate']['prices'].loc[years]

print(f"{'Year':>6}  {'Optimised':>12}  {'Fixed Path':>12}  {'Fixed Rate':>12}")
print("-" * 50)
for y in years[::5]:  # every 5th year
    print(f"{y:>6}  {prices_opt[y]:>12.2f}  {prices_fp[y]:>12.2f}  {prices_fr[y]:>12.2f}")

In [ ]:
# 3-way comparison chart
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=list(years), y=prices_opt.tolist(),
    name='Optimised', mode='lines',
    line=dict(color='#1f77b4', width=2)
))
fig.add_trace(go.Scatter(
    x=list(years), y=prices_fp.tolist(),
    name='Fixed Path ($50→$90 linear)', mode='lines',
    line=dict(color='#ff7f0e', width=2, dash='dash')
))
fig.add_trace(go.Scatter(
    x=list(years), y=prices_fr.tolist(),
    name='Fixed Rate (3% p.a.)', mode='lines',
    line=dict(color='#2ca02c', width=2, dash='dot')
))

fig.update_layout(
    title='Carbon price comparison: Optimised vs Fixed modes',
    xaxis_title='Year',
    yaxis_title='NZU price (2023 NZD / tCO₂-e)',
    template='plotly_white',
    legend=dict(x=0.02, y=0.98)
)
fig.show()
fig.write_image(str(output_dir / 'price_comparison.png'))

## 6. Results: supply–demand balance under fixed prices

When prices are fixed, the balance between supply and demand over the modelled period are no longer optimised. The gap is informative: it tells you whether the fixed price is above or below the market-clearing level.

In [ ]:
# Supply–demand balance for each scenario
def get_balance(scenario_name):
    model_result = NZU.results[scenario_name]['model']
    supply_total = model_result['supply']['total'].loc[years]
    demand       = model_result['demand'].loc[years]
    return (supply_total - demand)

balance_opt = get_balance('Optimised')
balance_fp  = get_balance('Fixed Path')
balance_fr  = get_balance('Fixed Rate')

print(f"{'Year':>6}  {'Optimised':>14}  {'Fixed Path':>14}  {'Fixed Rate':>14}")
print("-" * 58)
for y in years[::5]:
    print(f"{y:>6}  {balance_opt[y]:>14,.0f}  {balance_fp[y]:>14,.0f}  {balance_fr[y]:>14,.0f}")
print("\n(positive = surplus; negative = shortage)")

In [ ]:
# Supply–demand gap chart
fig2 = go.Figure()

for label, bal, colour in [
    ('Optimised',               balance_opt, '#1f77b4'),
    ('Fixed Path ($50→$90)',    balance_fp,  '#ff7f0e'),
    ('Fixed Rate (3% p.a.)',    balance_fr,  '#2ca02c'),
]:
    fig2.add_trace(go.Scatter(
        x=list(years), y=(bal / 1000).tolist(),
        name=label, mode='lines', line=dict(color=colour, width=2)
    ))

fig2.add_hline(y=0, line_dash='dash', line_color='grey', opacity=0.5)

fig2.update_layout(
    title='Supply–demand balance by scenario',
    xaxis_title='Year',
    yaxis_title='Balance (Mt CO₂-e)',
    template='plotly_white',
    legend=dict(x=0.02, y=0.98)
)
fig2.show()
fig2.write_image(str(output_dir / 'supply_demand_balance.png'))

## 7. Use-case: government price commitment

A common use-case is to model what happens if the government announces a specific price path — for instance, prices rising with a fixed nominal rate of 3% per year from the current NZU price.

Below we show how to quickly set this up and extract the key output: **what stockpile drawdown does that price path imply?**

In [ ]:
# Stockpile balance under fixed_rate scenario
fr_stk_results  = NZU.results['Fixed Rate']['model']['stockpile_results']
opt_stk_results = NZU.results['Optimised']['model']['stockpile_results']

fig3 = go.Figure()
for label, stk, colour in [
    ('Optimised',      opt_stk_results, '#1f77b4'),
    ('Fixed Rate 3%',  fr_stk_results,  '#2ca02c'),
]:
    fig3.add_trace(go.Scatter(
        x=list(years),
        y=(stk['stockpile_balance'].loc[years] / 1000).tolist(),
        name=label, mode='lines', line=dict(color=colour, width=2)
    ))

fig3.add_hline(y=0, line_dash='dash', line_color='grey', opacity=0.5)
fig3.update_layout(
    title='Stockpile balance: Optimised vs Fixed Rate 3%',
    xaxis_title='Year',
    yaxis_title='Stockpile balance (Mt NZUs)',
    template='plotly_white'
)
fig3.show()
fig3.write_image(str(output_dir / 'stockpile_comparison.png'))

## Summary

| Feature | `fixed_path` | `fixed_rate` | `optimised` |
|---|---|---|---|
| Price source | User-supplied pd.Series | User-supplied scalar growth rate | Solved by optimiser |
| Supply–demand balance | Not enforced | Not enforced | Enforced (gap → 0) |
| `price_change_rate` in result | `None` | User-supplied rate | Solved rate |
| `total_gap` in result | `None` | `None` | ~0 |
| Main use | Policy commitment, sensitivity | Rule-based scenarios | Market clearing |

### API recap

```python
# fixed_path
nzu.set_mode('pricing_mode', 'fixed_path', scenario='MyScenario')
nzu.fill('price_path', my_series, scenario='MyScenario')

# fixed_rate
nzu.set_mode('pricing_mode', 'fixed_rate', scenario='MyScenario')
nzu.fill('price_change_rate', 0.05, scenario='MyScenario')

nzu.run()
```